## NUMBERS ARE INCORRECT DO NOT USE FOR NOW

# Model Evaluation: Predicting Conflict Escalation

* **Model A:** Baseline (Tabular ACLED + Food + Rain)
* **Model B:** Baseline + Text embeddings

Text embeddings are integrated in four different ways:
* All events, no PCA (~789 features)
* All events, PCA (~71 features)
* Conflict-only events, no PCA (~789 features)
* Conflict-only events, PCA (~45 features)

This notebook compares the results from the best models where `k`=1.75 and the threshold fix has been applied, as dicussed in the methodology decisions notebook.

Each model configuration has a number of recorded results on:
* Train (2018-2022) where this is no civil war
* Onset (2023) including the three months where civil war esclated (April 2023)
* Active (2024-2025) where civil war had been active

In [1]:
import pandas as pd
import plotly.express as px

from utils.reporting import read_model_reports

MODELS = ["Model A",
 "Model B (conflict-only text PCA)",
 "Model B (conflict-only text non-PCA)",
 "Model B (all-event text non-PCA)",
 "Model B (all-event text PCA)"]


In [ ]:
def apply_final_config(df: pd.DataFrame, config) -> pd.DataFrame:
    """Filter the results for the decided config."""
    mask = pd.Series(True, index=df.index) 
    for col, val in config.items(): 
        mask &= (df[col] == val)

    food_ok = (df["include_food"] == False) | (df["price_recency"] == True) 
    mask &= food_ok 
    return df[mask].copy() 

part1_config = {
        "k": 1.75,
        "threshold_fix_applied": True,
    }
    

all_results = pd.read_csv("evaluation/sudan_results.csv")
results = apply_final_config(all_results, part1_config)

In [34]:
def variant_label(row):
    if row["include_text"] == False:
        return "model_a"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == True:
        return "conflict_pca"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == False:
        return "conflict_nopca"
    if row["conflict_only_embeddings"] == False and row["use_pca"] == True:
        return "all_pca"
    return "all_nopca"

def baseline_comparison_table(results, metrics):
    
    for m in metrics:
        results[m] = pd.to_numeric(results[m], errors="coerce")
        
    # Columns that define a "matched" comparison so it differs only when text is included 
    key_cols = ["include_food", "include_rain", "k", "event_col", "n_splits", "price_recency"]

    baseline = results[results["include_text"] == False].set_index(key_cols) # Baseline - no-text

    text_df = results[results["include_text"] == True].copy() # Text runs

    text_df["variant"] = text_df.apply(variant_label, axis=1)
    
        
    results_comparison = []
    for variant, grp in text_df.groupby("variant"):
        grp = grp.set_index(key_cols)
        matched = grp.join(baseline[metrics], rsuffix="_base", how="inner")

        # Calculate average n_predictors for the variant
        mean_preds = matched["n_predictors"].mean() if "n_predictors" in matched.columns else float('nan')

        row = {
            "variant": variant, 
            "n_matched_pairs": len(matched),
            "mean_n_predictors": round(mean_preds, 1)
        }
        
        for m in metrics:
            diff = matched[m] - matched[f"{m}_base"]
            row[f"percent_better_baseline_{m}"] = round((diff > 0).mean() * 100, 1)
            row[f"mean_diff_{m}"] = round(diff.mean(), 4)
            row[f"median_diff_{m}"] = round(diff.median(), 4)
            
        results_comparison.append(row)

    results_table = pd.DataFrame(results_comparison).set_index("variant")
    display(results_table)

# Part 1 - does text improve model performance?
Part 1 looks at the question of 'does text improve performance?' rather than comparing the final chosen best models. The following things have been fixed (see methdodology decisions for discussion):

* `k`=1.75 - this is fixed as it defines the prediction target.
* The threshold fix has been applied - this was a bug fix that has been resolved. 
* Food price recency flag has been included - this was a data porcessing issue that has been resolved.

The following implementation choices are allowed to roam in part 1:
* `n_splits`
* `event_col`
* The inclusion of additional strucutral variables (food/rain)
These parameters do not change the underlying task and only change what inputs the model draws on and how it is fit.


# 1. Training performance
The results on the cross-validation training splits give an indication on whether the models are overfitting to the training data and would therefore not be generalisable.

As the aim is to understand if adding text improves the baseline (structural features only), each baseline and text-added configuration was compared on training-CV AUPR and F1. The percentage of times that text improved each metric is recorded in the table below. The headline numbers indicate that text hurts model performance in training: on AUPR, only two of the four variants ever beat baseline, and only barely.

Conflict-only text, both with and without PCA, is tied as the best performing on training AUPR, each beating baseline in 2 of 16 comparisons (12.5%). Neither all-event variant beats baseline on AUPR in any comparison. On F1, conflict-only non-PCA is the clearer of the two, beating baseline in 4 of 16 comparisons (25%) against conflict-only PCA's 2 of 16 (12.5%), so it's the stronger overall performer of the two once both metrics are considered, but the AUPR result on its own does not distinguish between them.

In [35]:
train_metrics = ["train_cv_aupr", "train_cv_f1"]
baseline_comparison_table(results, train_metrics)

,n_matched_pairs,mean_n_predictors,percent_better_baseline_train_cv_aupr,mean_diff_train_cv_aupr,median_diff_train_cv_aupr,percent_better_baseline_train_cv_f1,mean_diff_train_cv_f1,median_diff_train_cv_f1
variant,,,,,,,,
all_nopca,16,790.5,0.0,-0.0395,-0.0374,0.0,-0.0526,-0.0473
all_pca,16,72.5,0.0,-0.0339,-0.0368,0.0,-0.0465,-0.0431
conflict_nopca,16,790.5,12.5,-0.0121,-0.0130,25.0,-0.0173,-0.0184
conflict_pca,16,46.5,12.5,-0.0133,-0.0160,12.5,-0.0183,-0.0182


# 2. Onset performance

Unlike training, text clearly helps at onset, but which text variant looks best depends on which metric is prioritised, and that split lines up with the text corpus each model draws on, not with PCA.

Both conflict-only variants outperform both all-event variants on AUPR:
* Conflict-only without PCA: 75.0% of 16 matched pairs beat baseline. With a mean AUPR diff of +0.017
* Conflict-only with PCA beats 68.8% of baseline models. With +0.009 mean AUPR diff

In contrast both all-event variants outperform both conflict-only variants on F1 (all-event non-PCA 87.5%, mean diff +0.032; all-event PCA 43.8%, mean diff -0.005). All-event text with PCA is the weakest text variant on AUPR specifically. It is  the only one with a negative mean difference (-0.0085), and the only one that fails to beat baseline in a majority of comparisons on either metric.

This isn't two variants pulling in different directions by chance. Conflict-only text runs a much more conservative operating point, it ranks escalation months well overall (hence the stronger AUPR across both its PCA and non-PCA forms), but is cautious about calling any specific month an escalation. All-event text runs the opposite, flagging more broadly, which drives its F1 advantage but also means two of all-event non-PCA's sixteen configurations collapse to predicting esclations for almost everything (onset_recall_class1 > 0.9). Given that a missed escalation is treated as the more costly error for an early-warning system in this project's framing, all-event text's recall-leaning approach is not obviously the wrong choice, but it should be considered a trade-off.

In [36]:
onset_metrics = ["onset_aupr", "onset_f1_class1"]
baseline_comparison_table(results, onset_metrics)

,n_matched_pairs,mean_n_predictors,percent_better_baseline_onset_aupr,mean_diff_onset_aupr,median_diff_onset_aupr,percent_better_baseline_onset_f1_class1,mean_diff_onset_f1_class1,median_diff_onset_f1_class1
variant,,,,,,,,
all_nopca,16,790.5,56.2,0.0038,0.0063,87.5,0.0318,0.0310
all_pca,16,72.5,43.8,-0.0085,-0.0054,43.8,-0.0050,-0.0022
conflict_nopca,16,790.5,75.0,0.0174,0.0202,25.0,-0.0217,-0.0362
conflict_pca,16,46.5,68.8,0.0090,0.0038,12.5,-0.0308,-0.0385


# 3. Active performance

Once the war is underway, the baseline model prevails almost everywhere. All four text variants now show negative mean AUPR differences against baseline.

Three of the four text variants perform very poorly during active conflict, and all show negative mean differences on both AUPR and F1. Once conflict has been running for months, region-month event counts stop being zero-inflated and the autoregressive/structural features are doing the actual work. The text embeddings aren't acting as a precursor signal the way they might pre-escalation, they're mostly extra dimensions that can cause overfitting.

During active conflict, the text model all-text with PCA is the best performing against baseline. During active conflict dimensionality reduction appears to specifically help text remain useful once conflict is underway. However, the same compression only works slightly on the conflict-only text model during active war. The pattern suggests it isn't PCA alone or text alone driving the active-period result, but the combination of the full event corpus (not just conflict events) compressed down to a smaller, less overfitting-prone feature set.

In [53]:
active_metrics = ["active_aupr", "active_f1_class1"]
baseline_comparison_table(results, active_metrics)

,n_matched_pairs,mean_n_predictors,percent_better_baseline_active_aupr,mean_diff_active_aupr,median_diff_active_aupr,percent_better_baseline_active_f1_class1,mean_diff_active_f1_class1,median_diff_active_f1_class1
variant,,,,,,,,
all_nopca,16,790.5,0.0,-0.0708,-0.0703,25.0,-0.0187,-0.0097
all_pca,16,72.5,25.0,-0.0087,-0.0107,50.0,-0.0133,0.0032
conflict_nopca,16,790.5,0.0,-0.0595,-0.0594,0.0,-0.1557,-0.1666
conflict_pca,16,46.5,6.2,-0.0373,-0.0359,6.2,-0.1188,-0.0960


# Part 2
Final model comparison.

## Setting included data

In part 1 dropping rain or dropping food, sometimes scores marginally higher for some variants. Deliberately not following that signal here is the point, if the final comparison quietly adopted whichever ablation flattered each model, the comparison would no longer be about text, it would be about whichever feature set happened to win, model by model. 

However for the final comparison between Model A and Model B variants, food and rain will be included in all. 

## Setting n_splits

`n_splits` controls the number of expanding-window cross-validation folds used during hyperparameter search (4 or 5, tested throughout the project).

Averaged across all five models, n_splits shifts mean AUPR by only 0.005-0.006 at every period (train 0.222 vs 0.216, onset 0.320 vs 0.328, active 0.189 vs 0.194), and the direction isn't even consistent — n=4 wins on train and
active, n=5 on onset. There's no simple decision on which number is better.

n_splits=5` is therefore fixed for Part 2, on the small but consistent AUPR advantage. A supporting check (holding food, rain, and event_col fixed and varying only n_splits) found the two all-event text variants are 1.5-1.7x more
sensitive to this choice than the baseline, while conflict-only text and the baseline itself are comparatively stable. Results for all-event text in Part 2 should be read with that in mind.

In [54]:
metrics = ["train_cv_aupr", "onset_aupr", "active_aupr"]

results.groupby("n_splits")[metrics].agg(["mean", "max", "count"]).round(4)

train_cv_aupr               onset_aupr               active_aupr  \
                  mean     max count       mean     max count        mean   
n_splits                                                                    
4               0.2220  0.2724    40     0.3200  0.3738    40      0.1892   
5               0.2162  0.2513    40     0.3284  0.3730    40      0.1936   

                        
             max count  
n_splits                
4         0.2785    40  
5         0.2562    40

In [55]:
results["variant"] = results.apply(variant_label, axis=1)
results.groupby(["n_splits", "variant"])[metrics].agg(["mean", "count"]).round(4)

train_cv_aupr       onset_aupr       active_aupr      
                                 mean count       mean count        mean count
n_splits variant                                                              
4        all_nopca             0.2059     8     0.3084     8      0.1483     8
         all_pca               0.2124     8     0.2974     8      0.2049     8
         conflict_nopca        0.2223     8     0.3462     8      0.1690     8
         conflict_pca          0.2289     8     0.3300     8      0.1930     8
         model_a               0.2403     8     0.3177     8      0.2307     8
5        all_nopca             0.1927     8     0.3387     8      0.1634     8
         all_pca               0.1975     8     0.3252     8      0.2310     8
         conflict_nopca        0.2311     8     0.3282     8      0.1654     8
         conflict_pca          0.2221     8     0.3277     8      0.1857     8
         model_a               0.2373     8     0.3220     8      0.2226     8

## Event type
`event_col` determines whether ACLED event features are chosen from the six events or the 25 sub-event types. Sub-events naturally give the model greater detail but this level of disaggregation reduces the number of positive instances of that sub-event. 

Interestingly three of five models (conflict-only non-PCA, conflict-only PCA, and Model A itself) differ in the best `event_type` depending on whether train-CV AUPR or onset AUPR is prioritised. 

To remain objective (rather than just picking the `event_type` that provides the best onset AUPR), `event_col` is selected using the column that most often increases train-CV AUPR, which is independent of the onset and active periods being reported on. 

`event_col` is set to sub_event_type throughout, the option train-CV AUPR narrowly favoured overall (21 of 40 paired comparisons).

In [56]:
results_fixed_data = results[(results["include_food"] == True) & (results["include_rain"] == True)]

for variant, group in results_fixed_data.groupby("variant"):
    g = group.set_index("event_col")[["train_cv_aupr", "onset_aupr"]]
    train_winner = g["train_cv_aupr"].idxmax()
    onset_winner = g["onset_aupr"].idxmax()
    agree = "AGREE" if train_winner == onset_winner else "DISAGREE"
    print(f"{variant}: train_cv prefers {train_winner}, onset prefers {onset_winner} -> {agree}")

all_nopca: train_cv prefers sub_event_type, onset prefers sub_event_type -> AGREE
all_pca: train_cv prefers event_type, onset prefers sub_event_type -> DISAGREE
conflict_nopca: train_cv prefers sub_event_type, onset prefers sub_event_type -> AGREE
conflict_pca: train_cv prefers event_type, onset prefers sub_event_type -> DISAGREE
model_a: train_cv prefers event_type, onset prefers sub_event_type -> DISAGREE


In [57]:
def event_col_preference_table(results, metrics):
    match_cols = ["include_food", "include_rain", "n_splits", "variant"]
    
    sub = results[results["event_col"] == "sub_event_type"].set_index(match_cols)
    evt = results[results["event_col"] == "event_type"].set_index(match_cols)
    
    rows = []
    for metric in metrics:
        paired = sub[[metric]].join(evt[[metric]], lsuffix="_sub", rsuffix="_evt", how="inner")
        for variant, grp in paired.reset_index().groupby("variant"):
            n_sub_wins = (grp[f"{metric}_sub"] > grp[f"{metric}_evt"]).sum()
            n_evt_wins = (grp[f"{metric}_evt"] > grp[f"{metric}_sub"]).sum()
            n_total = len(grp)
            rows.append({
                "variant": variant,
                "metric": metric,
                "n_pairs": n_total,
                "sub_event_type_wins": n_sub_wins,
                "event_type_wins": n_evt_wins,
            })
    
    table = pd.DataFrame(rows)
    
    totals = table.groupby("metric")[["n_pairs", "sub_event_type_wins", "event_type_wins"]].sum()
    totals["variant"] = "TOTAL"
    totals = totals.reset_index()
    print(totals)
    
    return pd.concat([table, totals], ignore_index=True)

event_col_preference_table(results, ["train_cv_aupr", "onset_aupr"])

          metric  n_pairs  sub_event_type_wins  event_type_wins variant
0     onset_aupr       40                   30               10   TOTAL
1  train_cv_aupr       40                   21               19   TOTAL


,variant,metric,n_pairs,sub_event_type_wins,event_type_wins
0,all_nopca,train_cv_aupr,8,3,5
1,all_pca,train_cv_aupr,8,6,2
2,conflict_nopca,train_cv_aupr,8,5,3
3,conflict_pca,train_cv_aupr,8,4,4
4,model_a,train_cv_aupr,8,3,5
5,all_nopca,onset_aupr,8,4,4
6,all_pca,onset_aupr,8,4,4
7,conflict_nopca,onset_aupr,8,6,2
8,conflict_pca,onset_aupr,8,8,0
9,model_a,onset_aupr,8,8,0


In [58]:
part2_config = {
        "k": 1.75,
        "threshold_fix_applied": True,
        "event_col": "sub_event_type",
        "include_food": True,
        "include_rain": True,
        "n_splits": 5
    }
    

In [59]:
final_results = apply_final_config(all_results, part2_config)
best_model_results, best_model_shap, best_model_onset_pred = read_model_reports(MODELS)

In [60]:
final_results

,run_name,run_id,data_version,threshold_fix_applied,include_food,include_rain,include_text,conflict_only_embeddings,use_pca,k,...,param_max_depth,param_max_delta_step,param_learning_rate,param_gamma,param_colsample_bytree,param_colsample_bylevel,param_k,param_n_splits,param_event_col,price_recency
1120,acled_sub_food_rain_text_conflict_price-recenc...,54dbf812379f4a138663ec1fd595dec5,acled_sub_food_rain_text_conflict_price-recenc...,True,True,True,True,True,True,1.75,...,7,5,0.01,0,0.8,0.6,1.75,5,sub_event_type,True
1121,acled_sub_food_rain_text_conflict_price-recenc...,70d1da74cc8c48729388a79a06ac0683,acled_sub_food_rain_text_conflict_price-recenc...,True,True,True,True,True,False,1.75,...,3,5,0.03,1,0.6,0.8,1.75,5,sub_event_type,True
1122,acled_sub_food_rain_text_all_price-recency_thr...,a2848e660df443fc9ff0027d29336dfa,acled_sub_food_rain_text_all_price-recency_thr...,True,True,True,True,False,True,1.75,...,7,5,0.01,0,0.8,0.6,1.75,5,sub_event_type,True
1123,acled_sub_food_rain_text_all_price-recency_thr...,f1b4d8d9ad564bb1a23010434b69cd07,acled_sub_food_rain_text_all_price-recency_thr...,True,True,True,True,False,False,1.75,...,5,1,0.01,5,0.8,1.0,1.75,5,sub_event_type,True
1128,acled_sub_food_rain_price-recency_threshold_ch...,c9a5905ccbe44b589f680a60053448d1,acled_sub_food_rain_price-recency_threshold_ch...,True,True,True,False,False,False,1.75,...,7,5,0.01,0,0.8,0.6,1.75,5,sub_event_type,True


# IGNORE ANYTHING BELOW HERE

# 4. Regional onset performance

Looking at training and onset performance together, it might be logical to conclude that Conflict-only text and Non-PCA is one of the strongest models based on aggregate onset AUPR. However, this doesn't hold once you break it down by region. The 2023 escalation wasn't spread evenly, most of it was concentrated in a handful of regions, so what matters more than the aggregate score is whether a model actually catches escalation in the regions where it happened.

The table below looks at recall specifically in the regions with the most true escalation months during the onset window, and at Khartoum on its own, since that's where the April 2023 fighting actually broke out.

This is run on the best config for each of the variants. 

In [8]:
KEY_REGIONS = [
    "Khartoum", "North Darfur", "South Darfur", "West Darfur",
    "Central Darfur", "East Darfur", "West Kordofan", "South Kordofan",
]

def region_recall_table(onset_pred_df, regions, model_col="model"):
    rows = []
    for model, grp in onset_pred_df.groupby(model_col):
        key_region_rows = grp[grp["region"].isin(regions)]
        khartoum_rows = grp[grp["region"] == "Khartoum"]

        def recall(sub):
            n_true = sub["y_true"].sum()
            n_caught = ((sub["y_true"] == 1) & (sub["y_pred"] == 1)).sum()
            return n_caught, n_true, (n_caught / n_true if n_true else float("nan"))

        n_caught, n_true, key_recall = recall(key_region_rows)
        kh_caught, kh_true, kh_recall = recall(khartoum_rows)

        rows.append({
            "model": model,
            "key_regions_caught": f"{n_caught}/{n_true}",
            "key_regions_recall": round(key_recall, 3),
            "khartoum_caught": f"{kh_caught}/{kh_true}",
            "khartoum_recall": round(kh_recall, 3),
        })
    return pd.DataFrame(rows).set_index("model")

region_recall_table(best_model_onset_pred, KEY_REGIONS)

,key_regions_caught,key_regions_recall,khartoum_caught,khartoum_recall
model,,,,
Model A,15/27,0.556,1/5,0.2
Model B (all-event text PCA),7/27,0.259,0/5,0.0
Model B (all-event text non-PCA),19/27,0.704,2/5,0.4
Model B (conflict-only text PCA),7/27,0.259,0/5,0.0
Model B (conflict-only text non-PCA),4/27,0.148,0/5,0.0


Conflict-only non-PCA isn't the best variant here. It misses 4 esclations in Khartoum entirely and comes only just above the baseline on recall across the other key regions too. All-event text without PCA does the opposite: it beats both the baseline and conflict-only PCA on regional recall, including catching escalation months in Khartoum that neither of the other two models flag.

This is the same trade-off that shows up in the aggregate onset F1 numbers above (all-event text has the stronger F1, conflict-only PCA the stronger AUPR), just made concrete at the region level. AUPR is measuring ranking quality across every possible threshold, so a model can score well there while still, at the actual threshold it's deployed with, being too conservative to flag the events that matter most. The conflict-only PCA model's overalll onset recall sits well below the other two models (see Section 2), and that is most present in exactly the regions where the war started.

For an early-warning use case, missing Khartoum isn't a small issue, it is fundamentally failing to achieve its purpose. On that basis, all-event text without PCA is the better candidate for onset detection specifically, even though it's the weaker performer on train-CV and on aggregate AUPR. However, this is a very small sample size of actual escalation. This also points to a broader limitation of this sample size. The onset finding rests on a single historical case as Sudan's 2023 escalation is the only event of its kind in this dataset, so the claim that text helps at onset should be read as "text helped for this one escalation", not as a validated general property. 

# 5. Feature importance: why the pattern flips

SHAP scores help identify why the pattern changes. SHAP importance was computed separately on the training, onset, and active periods to test whether text's contribution shifts once conflict is underway. 

Text embeddings' share of importance does not shrink during active war, it increases in all four text variants (e.g. all-event non-PCA rises from 48.9% of importance in training to 56.3% during active conflict). Rather than the model correctly deprioritising text once its precursor value is no longer relevant, it appears to rely on text more heavily during exactly the period where that reliance coincides with the sharpest drop in predictive performance (Section 4). This suggests the active-period AUPR and F1 declines are indicative of the model continuing to draw on text-derived signal that does not generalise to active-conflict dynamics. 

Consistent with the training-CV finding in Section 1 that text-derived importance during fitting does not guarantee that signal transfers to unseen periods. 


In [9]:
shap_by_category = (
    best_model_shap.groupby(["model", "dataset", "category"])["mean_abs_shap"]
    .sum()
    .reset_index()
)

shap_by_category["total_SHAP"] = shap_by_category.groupby(["model", "dataset"])["mean_abs_shap"].transform("sum")
shap_by_category["% Importance"] = (
    shap_by_category["mean_abs_shap"] / shap_by_category["total_SHAP"] * 100
)

fig = px.bar(
    shap_by_category,
    x="model",
    y="% Importance",
    color="category",        
    facet_row="dataset",
    category_orders={"dataset": ["train", "onset", "active"]},
    title="SHAP feature importance by category (Train vs Onset vs Active)",
    text_auto=".1f",
    color_discrete_sequence=px.colors.qualitative.Bold,
    height=900,   
)

fig.update_layout(
    xaxis_title="",
    legend_title_text="Feature Category",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=50, b=50, l=50, r=50)
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].capitalize()))

# Add light gridlines and standardise y-axis titles for all rows
fig.update_yaxes(
    title_text="Relative Importance (%)", 

)
fig.update_xaxes(
    showline=True, 
    linewidth=1, 
    linecolor='black'
)

fig.show()

# 6. Ethiopia results to do

# 7. Summary

Pulling the three sections together:

- **Train-CV (Section 1):** None of the text models outperform the baseline model in terms of generalisation. Conflict-only text without PCA is the least weak of four underperforming options (beating baseline on 25% of matched pairs), and that's plausibly explained by 2018-2022 not containing anything resembling the 2023 escalation.
- **Onset, aggregate (Section 2):** All-event text without PCA is the strongest text variant outright, beating baseline on both AUPR and F1 in 75% of matched pairs. Conflict-only text (both variants) shows a large gap between its AUPR performance (62.5%) and F1 performance (12.5-25%), suggesting it ranks escalation months reasonably well but is too conservative at the threshold actually deployed.
- **Onset, regional (Section 3):** Conflict-only PCA's AUPR advantage doesn't match the geography of the war. It misses Khartoum (the centre point of the esclation) and underperforms the baseline in the regions that mattered most. All-event text is the model that actually catches the escalation where it happened. However, this is a very small sample size. 
- **Active (Section 4):** TThe baseline wins clearly for three of the four text variants, and decisively so for conflict-only non-PCA. All-event text with PCA is the exception, running roughly level with or marginally ahead of baseline, which suggests the active-period story is about dimensionality and corpus breadth together, not simply that text fails when conflict starts.

Given the purpose of this project is to understand if adding text to a baseline model improves the prediction of conflict escalation, all-event text without PCA remains the strongest candidate for onset detection specifically, now winning cleanly on both AUPR and F1 rather than needing a metric-specific argument. All-event text with PCA is worth retaining as the candidate for active-period monitoring, since it's the only text variant that doesn't lose ground to the baseline once conflict is underway. Conflict-only text, in either form, is the weakest performer across all three periods and is hardest to justify keeping in the final model.


In [10]:
k175 = sudan[(sudan["k"] == 1.75) & (sudan["threshold_fix_applied"] == True)]

k175.groupby("n_splits")["onset_aupr"].agg(["mean", "max", "count"]).round(4)

NameError: name 'sudan' is not defined